In [ ]:
from brainspace.mesh.mesh_io import read_surface
from brainspace.utils.parcellation import map_to_labels
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.stats import pearsonr
from surfplot import Plot

import cmasher as cmr
import glob
import matplotlib.pyplot as plt
import matplotlib as mpl
import natsort
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import pingouin as pg
import seaborn as sns
import SUITPy.flatmap as flatmap
import warnings
warnings.filterwarnings('ignore')

def _align_labels_to_atlas(x, source_labels, target_labels):
    target = np.unique(target_labels)[1:]
    df1 = pd.DataFrame(target, index=target)
    df2 = pd.DataFrame(x, index=source_labels)
    return pd.concat([df1, df2], axis=1).iloc[:, 1:].values
    
def weights_to_vertices(data, target, labels=None):
    if isinstance(target, str): 
        vertices = nib.load(target).get_fdata().ravel()
    else:
        vertices = target.ravel()
    if labels is not None:
        data = _align_labels_to_atlas(data, labels, vertices)
    mask = vertices != 0
    map_args = dict(target_lab=vertices, mask=mask, fill=np.nan)
    if (len(data.shape) == 1) or (data.shape[1] == 1):
        weights = map_to_labels(data.ravel(),  **map_args)
    else:
        weights = [map_to_labels(x, **map_args) for x in data.T]
    return weights


def get_surfaces(style='inflated', load=True):
    surf_path = '../utils/surfaces'
    surfaces = get_files([surf_path, f'*.{style}_*'])
    if load:
        surfs = [read_surface(i) for i in surfaces]
        return dict(zip(['lh', 'rh'], surfs))
    else:
        return surfaces
        
def get_files(pattern, force_list=False):
    if isinstance(pattern, list):
        pattern = os.path.join(*pattern)
    files = natsort.natsorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Pattern could not detect file(s): {pattern}")
    if (not force_list) and (len(files) == 1):
        return files[0]
    else:
        return files

def plot_subcortex(data, cmap, color_range, save_dir, surf_file='../utils/atlases/subcorticalSurf.pkl'):
    with open(surf_file, 'rb') as f:
        VL, VR = pickle.load(f)

    nROI = len(VL)

    cmap = cmap.copy()
    cmap.set_bad(color='grey')   
    cmap.set_under(color='grey')

    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(141, projection='3d') 
    ax2 = fig.add_subplot(142, projection='3d')  
    ax3 = fig.add_subplot(143, projection='3d') 
    ax4 = fig.add_subplot(144, projection='3d')  

    vmin, vmax = color_range
    def val_to_color(val):
        if val is None or np.isnan(val):
            return cmap(np.nan)
        frac = (val - vmin) / float(vmax - vmin)
        return cmap(frac)

    coords_ax1, coords_ax2, coords_ax3, coords_ax4 = [], [], [], []

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]    # left ROIs in second half
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax1.add_collection3d(mesh)
            coords_ax1.append(coords)

    ax1.view_init(elev=0, azim=0)
    ax1.set_box_aspect((1, 1, 1))
    ax1.set_axis_off()

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax2.add_collection3d(mesh)
            coords_ax2.append(coords)

    ax2.view_init(elev=0, azim=180)
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]           # right ROIs in first half
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax3.add_collection3d(mesh)
            coords_ax3.append(coords)

    ax3.view_init(elev=0, azim=0)
    ax3.set_box_aspect((1, 1, 1))
    ax3.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax4.add_collection3d(mesh)
            coords_ax4.append(coords)

    ax4.view_init(elev=0, azim=180)
    ax4.set_box_aspect((1, 1, 1))
    ax4.set_axis_off()

    if coords_ax1:
        all_coords_1 = np.concatenate(coords_ax1, axis=0)
        ax1.auto_scale_xyz(all_coords_1[:, 0], all_coords_1[:, 1], all_coords_1[:, 2])

    if coords_ax2:
        all_coords_2 = np.concatenate(coords_ax2, axis=0)
        ax2.auto_scale_xyz(all_coords_2[:, 0], all_coords_2[:, 1], all_coords_2[:, 2])

    if coords_ax3:
        all_coords_3 = np.concatenate(coords_ax3, axis=0)
        ax3.auto_scale_xyz(all_coords_3[:, 0], all_coords_3[:, 1], all_coords_3[:, 2])

    if coords_ax4:
        all_coords_4 = np.concatenate(coords_ax4, axis=0)
        ax4.auto_scale_xyz(all_coords_4[:, 0], all_coords_4[:, 1], all_coords_4[:, 2])

    plt.tight_layout()
    plt.savefig(f"{save_dir}", dpi=800, transparent=True)
    plt.close()

def plot_cerebellum(df, value_col, color_range, cmap, save_path, label_file='../utils/atlases/atl-NettekovenAsym32_dseg.label.gii'):
    cerebellum_labels = ["region1", "region2", "region3", "region4", "region5", "region6", "region7", "region8", "region9", "region10", 
                         "region11", "region12", "region13", "region14", "region15", "region16", "region17", "region18", "region19", 
                         "region20", "region21", "region22", "region23", "region24", "region25", "region26", "region27", "region28", 
                         "region29", "region30", "region31", "region32"]

    vmin, vmax = color_range
    missing_val = vmin - 1.0
    roi_values = np.full(len(cerebellum_labels), np.nan, dtype=float)
    roi_map = dict(zip(df["roi"].astype(str), df[value_col].astype(float)))

    for i, label in enumerate(cerebellum_labels):
        if label in roi_map:
            roi_values[i] = roi_map[label]

    labels = nib.load(label_file)
    label_data = labels.darrays[0].data.astype(int)
    num_vertices = label_data.shape[0]
    data = np.full(num_vertices, missing_val, dtype=float)  

    for i, roi_value in enumerate(roi_values):
        mask = (label_data == (i + 1))
        if not np.any(mask):
            continue

        if np.isnan(roi_value):
            data[mask] = missing_val
        else:
            data[mask] = roi_value

    cmap = cmap.copy()
    cmap.set_under('grey')

    ax = flatmap.plot(data, overlay_type='func', cmap=cmap, cscale=(vmin, vmax), new_figure=True, colorbar=True, bordercolor='w')

    plt.savefig(save_path, dpi=800, transparent=True)
    plt.close()

def yeo_cmap():
    cmap = {'cerebellum': (0, 0, 0), 'subcortex': (77, 77, 77), 'Vis': (119, 20, 140), 'SomMot': (70, 126, 175), 
            'DorsAttn': (0, 117, 7), 'SalVentAttn': (195, 59, 255), 'Limbic': (219, 249, 165), 'Cont': (230, 149, 33), 
            'Default': (205, 65, 80)}
    cmap = {k: np.array(v) / 255 for k, v in cmap.items()}
    return cmap

def parse_roi_names(df, col='roi'):
    cerebellum_labels = ["M1L", "M2L", "M3L", "M4L", "A1L", "A2L", "A3L", "D1L", "D2L", "D3L", "D4L",
                         "S1L", "S2L", "S3L", "S4L", "S5L", "M1R", "M2R", "M3R", "M4R", "A1R", "A2R", 
                         "A3R", "D1R", "D2R", "D3R", "D4R", "S1R", "S2R", "S3R", "S4R", "S5R"]
    cerebellum_map = {f"region{i+1}": lab for i, lab in enumerate(cerebellum_labels)}
    cerebellum_index = {f"region{i+1}": i+1 for i in range(32)}  

    subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                       "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                       "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                       "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
    subcortex_index = {lab: i+1 for i, lab in enumerate(subcortex_order)}  

    def parse_single_roi(roi_value):
        if pd.isna(roi_value):
            return pd.Series([None, None, None, None], index=['hemi','network','name','roi_renamed'])
        roi_str = str(roi_value).strip()

        if roi_str in cerebellum_map:
            label = cerebellum_map[roi_str]      
            hemi = "RH" if label.endswith("R") else "LH"
            network = "cerebellum"
            name_idx = cerebellum_index[roi_str]     
            return pd.Series([hemi, network, name_idx, label], index=['hemi','network','name','roi_renamed'])

        if roi_str in subcortex_index:
            hemi = "RH" if roi_str.endswith('-rh') else "LH"
            network = "subcortex"
            name_idx = subcortex_index[roi_str]    
            return pd.Series([hemi, network, name_idx, roi_str], index=['hemi','network','name','roi_renamed'])

        parts = roi_str.split('_', 3)
        if len(parts) == 4:
            hemi, network, name = parts[1], parts[2], parts[3]
        else:
            hemi, network, name = None, None, roi_str
        return pd.Series([hemi, network, name, roi_str], index=['hemi','network','name','roi_renamed'])

    parsed = df[col].apply(parse_single_roi)
    df[['hemi','network','name']] = parsed[['hemi','network','name']]
    df[col] = parsed['roi_renamed']
    return df

def plot_seed_eccentricity(df, seed, base_color):
    epoch_colors = {"base": "grey", "early": "grey", "late": "grey"}
    
    df["distance"] = pd.to_numeric(df["distance"], errors="coerce")
    df["epoch"] = df["epoch"].astype(str).str.lower().str.strip()
    df = df[df["epoch"].isin(["base", "early", "late"])].copy()
    df = df.dropna(subset=["distance"]).copy()

    subject_points = (df.groupby(["sub", "epoch"], as_index=False)["distance"].mean())
    subject_points["epoch"] = pd.Categorical(subject_points["epoch"], categories=["base", "early", "late"], ordered=True)
    subject_points = subject_points.sort_values(["sub", "epoch"])
    subject_points["epoch_num"] = subject_points["epoch"].cat.codes.astype(float)

    subs = subject_points["sub"].unique()
    subj_jitter = pd.Series(np.random.uniform(-0.15, 0.15, size=len(subs)), index=subs)

    fig, ax = plt.subplots(figsize=(3.2, 4.0))

    for sub, g in subject_points.groupby("sub"):
        g = g.sort_values("epoch_num")
        x = g["epoch_num"].to_numpy(dtype=float) + float(subj_jitter.loc[sub])
        y = g["distance"].to_numpy(dtype=float)

        point_colors = [epoch_colors.get(str(e), "gray") for e in g["epoch"].astype(str)]
        ax.scatter(x, y, c=point_colors, s=50, alpha=0.5, edgecolor="white", linewidth=0.6, zorder=3)

    mean_line = (subject_points.groupby("epoch_num", as_index=False)["distance"].mean().sort_values("epoch_num"))
    ax.plot(mean_line["epoch_num"], mean_line["distance"], color="black", linewidth=4, marker="o", markersize=11,
            markerfacecolor=base_color, zorder=10)
    ax.set_xticks([0, 1, 2])
    ax.yaxis.set_major_locator(MultipleLocator(0.5))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"../figures/seed/seed_eccentricity_{seed}.png", dpi=800, transparent=True)
    plt.close()

def connect_seed(cmats, seed_region):
    results = []
    for c in cmats:
        cmat = pd.read_table(c, index_col=0)
        res = pd.DataFrame(cmat.loc[seed_region].reset_index().values, columns=['roi', 'r'])
        res['roi_ix'] = np.arange(464) + 1
        fname = os.path.basename(c)
        res['sub'] = fname.split('_')[0]
        res['epoch'] = fname.split('_')[-1].split('.')[0]
        task_name = fname.split('_')[-4].split('.')[0]
        res['task'] = "rl" if "RL" in task_name else "ml"
        results.append(res)
    connectivity = pd.concat(results)
    connectivity['r'] = connectivity['r'].astype(float)
    return connectivity
    
def seed_analysis(cmats, epochs, seed):
    os.makedirs("../output/seed", exist_ok=True)
    
    connectivity = connect_seed(cmats, seed)
    df = connectivity.query("epoch in @epochs")
    df.to_csv(f"../output/seed/seed_{epochs[1]}_vs_{epochs[0]}_{seed}.tsv", sep='\t', index=False)

    res = df.groupby(['roi', 'roi_ix'], sort=False).apply(pg.pairwise_tests, dv='r', within='epoch', subject='sub').reset_index()
    res['T'] = -res['T']
    res['sig'] = (res['p-unc'] < .05).astype(float)

    return res, seed
    
def plot_seed_analysis():
    cmats = get_files("../output/connectivity/sub*/*tsv*")
    cmats = [x for x in cmats if "rest" not in x.lower()]
    
    os.makedirs("../figures/network-seed", exist_ok=True)

    for seed_region, col in zip(["7Networks_RH_Limbic_OFC_1", "7Networks_RH_SalVentAttn_FrOperIns_5" , "7Networks_LH_Default_Par_2", "7Networks_RH_DorsAttn_FEF_1"], ["#E1F8AD", "#B649F6", "#C1575D", "#337423"]):
        for epoch in [["early", "base"], ["late", "early"], ["late", "base"]]:
    
            suffix = f'{epoch[1]}_vs_{epoch[0]}'
            res, seed = seed_analysis(cmats, epoch, seed_region)
            res.to_csv(f"../output/seed/tvals_{epoch[1]}_vs_{epoch[0]}_{seed_region}.tsv", sep='\t', index=False)

            #1.1 Plot Cortex
            vmax = 4.9

            cortex = res[(res["roi"].astype(str).str.startswith("7Net"))]
            surfaces = get_surfaces()
            cmap = cmr.get_sub_cmap('bwr', 0.01, 0.99)
            seed_cmap = LinearSegmentedColormap.from_list('regions', [col, 'k'], N=2)
            seed = res[res['roi'] == seed_region]
            x = weights_to_vertices(cortex["T"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", [x - 64 for x in cortex["roi_ix"]])
            z = weights_to_vertices(seed["T"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", [x - 64 for x in seed["roi_ix"]])
            p = Plot(surfaces['lh'], surfaces['rh'])
            p.add_layer(x, cmap=cmap, color_range=(-vmax, vmax), cbar=False)
            p.add_layer(z, cmap=seed_cmap, cbar=False)
            p.add_layer(z, cmap='binary_r', cbar=False, as_outline=True)
            fig = p.build(colorbar=False)
            plt.savefig(f"../figures/seed/{seed_region}_{suffix}_cortex.png", dpi=800, transparent=True)
            plt.close()
   
            subcortex = res[res["roi"].astype(str).str.endswith(("-lh", "-rh"))].sort_values(by="roi_ix")
            subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                               "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                               "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                               "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
            roi_to_idx_sc = {roi: i for i, roi in enumerate(subcortex_order)}
            f_sc = subcortex[subcortex["roi"].isin(subcortex_order)].set_index("roi")["T"]
            data_sc = np.full(len(subcortex_order), np.nan, dtype=float)
            for roi, F in f_sc.items():
                idx = roi_to_idx_sc[roi]
                if np.isfinite(F):
                    data_sc[idx] = float(F)
            plot_subcortex(data_sc, cmr.get_sub_cmap('bwr', 0.01, 0.99), (-vmax, vmax), save_dir=f"../figures/seed/{seed_region}_{suffix}_subcortex.png")
    
            #2.3 Cerebellum
            cerebellum = res[res["roi"].astype(str).str.startswith(("region"))].sort_values(by="roi_ix")
            plot_cerebellum(df=cerebellum, value_col="T", color_range=[-vmax, vmax], cmap=cmr.get_sub_cmap('bwr', 0.01, 0.99), save_path=f"../figures/seed/{seed_region}_{suffix}_cerebellum.png") 

    for seed_region, col in zip(["7Networks_LH_Limbic_OFC_1", "7Networks_LH_SalVentAttn_FrOperIns_5" , "7Networks_RH_Default_Par_2", "7Networks_LH_DorsAttn_FEF_1"], ["#E1F8AD", "#B649F6", "#C1575D", "#337423"]):
        for epoch in [["early", "base"], ["late", "early"], ["late", "base"]]:
    
            suffix = f'{epoch[1]}_vs_{epoch[0]}'
            res, seed = seed_analysis(cmats, epoch, seed_region)
            res.to_csv(f"../output/seed/tvals_{epoch[1]}_vs_{epoch[0]}_{seed_region}.tsv", sep='\t', index=False)

            #1.1 Plot Cortex
            vmax = 4.9

            cortex = res[(res["roi"].astype(str).str.startswith("7Net"))]
            surfaces = get_surfaces()
            cmap = cmr.get_sub_cmap('bwr', 0.01, 0.99)
            seed_cmap = LinearSegmentedColormap.from_list('regions', [col, 'k'], N=2)
            seed = res[res['roi'] == seed_region]
            x = weights_to_vertices(cortex["T"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", [x - 64 for x in cortex["roi_ix"]])
            z = weights_to_vertices(seed["T"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", [x - 64 for x in seed["roi_ix"]])
            p = Plot(surfaces['lh'], surfaces['rh'])
            p.add_layer(x, cmap=cmap, color_range=(-vmax, vmax), cbar=False)
            p.add_layer(z, cmap=seed_cmap, cbar=False)
            p.add_layer(z, cmap='binary_r', cbar=False, as_outline=True)
            fig = p.build(colorbar=False)
            plt.savefig(f"../figures/seed/{seed_region}_{suffix}_cortex.png", dpi=800, transparent=True)
            plt.close()
   
            subcortex = res[res["roi"].astype(str).str.endswith(("-lh", "-rh"))].sort_values(by="roi_ix")
            subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                               "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                               "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                               "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
            roi_to_idx_sc = {roi: i for i, roi in enumerate(subcortex_order)}
            f_sc = subcortex[subcortex["roi"].isin(subcortex_order)].set_index("roi")["T"]
            data_sc = np.full(len(subcortex_order), np.nan, dtype=float)
            for roi, F in f_sc.items():
                idx = roi_to_idx_sc[roi]
                if np.isfinite(F):
                    data_sc[idx] = float(F)
            plot_subcortex(data_sc, cmr.get_sub_cmap('bwr', 0.01, 0.99), (-vmax, vmax), save_dir=f"../figures/seed/{seed_region}_{suffix}_subcortex.png")
    
            #2.3 Cerebellum
            cerebellum = res[res["roi"].astype(str).str.startswith(("region"))].sort_values(by="roi_ix")
            plot_cerebellum(df=cerebellum, value_col="T", color_range=[-vmax, vmax], cmap=cmr.get_sub_cmap('bwr', 0.01, 0.99), save_path=f"../figures/seed/{seed_region}_{suffix}_cerebellum.png") 

def load_three_epochs(seed_network):
    df_be = pd.read_csv(f"../output/seed/seed_base_vs_early_{seed_network}.tsv", sep="\t")
    df_el = pd.read_csv(f"../output/seed/seed_early_vs_late_{seed_network}.tsv", sep="\t")
    df_bl = pd.read_csv(f"../output/seed/seed_base_vs_late_{seed_network}.tsv", sep="\t")
    return pd.concat([df_be, df_el, df_bl], ignore_index=True)

def plot_barplots_network(df, seed, network_order=("subcortex", "cerebellum", "Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default")):
    epoch_colors = {"base": "yellow", "early": "red", "late": "royalblue"}

    df["epoch"] = df["epoch"].astype(str).str.lower().str.strip()
    df = df.groupby(["epoch", "network"], as_index=False)["r"].mean()
    wide = (df.pivot(index="network", columns="epoch", values="r").reindex(index=list(network_order)))
    wide = wide[list(("base", "early", "late"))]

    x = np.arange(len(network_order))
    width = 0.25
    offsets = {"base": -width, "early": 0.0, "late": +width}
    
    fig, ax = plt.subplots(figsize=(5, 4))
    for e in ("base", "early", "late"):
        ax.bar(x + offsets[e], wide[e].to_numpy(dtype=float), width=width, label=e, color=epoch_colors.get(e, "gray"),
               edgecolor="black", linewidth=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(network_order, rotation=35, ha="right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../figures/seed/network_barplot_{seed}.png", dpi=800, transparent=True)
    plt.close()
    
def plot_polarplot_seed(data, seed, epochs=("base", "early", "late")):
    data = data.groupby(["epoch", "network"], as_index=False)["r"].mean()

    colors = {"base": "yellow", "early": "red", "late": "royalblue"} 
    networks = ["subcortex", "cerebellum", "Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default"]

    data = data[data["network"].isin(networks)]
    data = data.groupby(["epoch", "network"], as_index=False)["r"].mean()
    pivot = (data.pivot(index="epoch", columns="network", values="r").reindex(columns=networks))

    angles = np.linspace(0, 2*np.pi, len(networks), endpoint=False)
    angles_closed = np.r_[angles, angles[0]]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
    
    for epoch in epochs:
        values = np.nan_to_num(pivot.loc[epoch].to_numpy(dtype=float), nan=0.0)
        values_closed = np.r_[values, values[0]]
        c = colors.get(epoch, "gray")

        ax.plot(angles_closed, values_closed, linewidth=2, label=epoch, color=c)
        ax.fill(angles_closed, values_closed, color=c, alpha=0.20)
        ax.scatter(angles, values, s=45, facecolor="white", edgecolor=c, linewidth=1.2, zorder=3)

    ax.set_xticks(angles)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", pad=20)

    for label, angle in zip(ax.get_xticklabels(), angles):
        angle_deg = np.degrees(angle)
        if 90 < angle_deg < 270:
            label.set_rotation(angle_deg + 180)
            label.set_horizontalalignment("right")
        else:
            label.set_rotation(angle_deg)
            label.set_horizontalalignment("left")
        label.set_rotation_mode("anchor")
        label.set_verticalalignment("center")

    rmin, rmax = ax.get_ylim()
    step = 0.1
    rticks = np.arange(np.floor(rmin/step)*step, np.ceil(rmax/step)*step + step, step)
    ax.set_yticks(rticks)
    for label in ax.get_yticklabels():
        label.set_fontname("Times New Roman")
        label.set_fontsize(25)
        
    ax.yaxis.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
    ax.xaxis.grid(False)

    plt.tight_layout()
    plt.savefig(f"../figures/seed/seed_polarplot_{seed}.png", dpi=800, transparent=True)
    plt.close()

def plot_polarplot_seed_diffs(data, seed, networks=None):
    if networks is None:
        networks = ["subcortex", "cerebellum", "Vis", "SomMot", "DorsAttn",
                    "Limbic", "Cont", "SalVentAttn", "Default"]

    d = (data[data["network"].isin(networks)].groupby(["epoch", "network"], as_index=False)["r"].mean())
    pivot = (d.pivot(index="epoch", columns="network", values="r").reindex(index=["base", "early", "late"], columns=networks))

    EB = pivot.loc["early"] - pivot.loc["base"]   # Early - Base
    LE = pivot.loc["late"]  - pivot.loc["early"]  # Late - Early
    LB = pivot.loc["late"]  - pivot.loc["base"]   # Late - Base

    angles = np.linspace(0, 2*np.pi, len(networks), endpoint=False)
    angles_closed = np.r_[angles, angles[0]]

    def _polar(ax, series, color, label):
        vals = series.to_numpy(dtype=float)
        vals = np.nan_to_num(vals, nan=0.0) 
        vals_closed = np.r_[vals, vals[0]]

        ax.plot(angles_closed, vals_closed, lw=2, color=color, label=label)
        ax.fill(angles_closed, vals_closed, color=color, alpha=0.20)
        ax.scatter(angles, vals, s=45, facecolor="white", edgecolor=color, linewidth=1.2, zorder=3)

    def _format_ax(ax):
        ax.set_xticks(angles)
        ax.set_xticklabels([])    
        ax.tick_params(axis="x", pad=20)

        rmin, rmax = ax.get_ylim()
        step = 0.1
        rticks = np.arange(np.floor(rmin/step)*step, np.ceil(rmax/step)*step + step, step)
        ax.set_yticks(rticks)
        for lab in ax.get_yticklabels():
            lab.set_fontname("Times New Roman")
            lab.set_fontsize(25)

        ax.yaxis.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
        ax.set_ylim(-0.1, 0.1)
        ax.xaxis.grid(False)
        ax.plot(np.linspace(0, 2*np.pi, 400), np.zeros(400), color="black", linewidth=1.5, zorder=2)

    fig1, ax1 = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})

    _polar(ax1, EB, color="deeppink", label="Early - Base")
    _polar(ax1, LE, color="orange",   label="Late - Early")

    _format_ax(ax1)
    
    plt.tight_layout()
    plt.savefig(f"../figures/seed/seed_polarplot_diffs1_{seed}.png", dpi=800, transparent=True)
    plt.close(fig1)

    fig2, ax2 = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
    _polar(ax2, LE, color="orange",        label="Late - Early")
    _polar(ax2, LB, color="plum",          label="Late - Base")  # light purple
    _format_ax(ax2)
    plt.tight_layout()
    plt.savefig(f"../figures/seed/seed_polarplot_diffs2_{seed}.png", dpi=800, transparent=True)
    plt.close(fig2)
    
def main():
    plot_seed_analysis()
    
    epoch_res = pd.read_csv("../output/subject_gradients.tsv", sep="\t")
    for seed, col in zip(["7Networks_LH_Limbic_OFC_1", "7Networks_LH_SalVentAttn_FrOperIns_5" , "7Networks_RH_Default_Par_2", "7Networks_LH_DorsAttn_FEF_1"], ["#E1F8AD", "#B649F6", "#C1575D", "#337423"]):
        epoch_cluster = (epoch_res.groupby(['sub', 'roi', 'epoch', 'network'], as_index=False)['distance'].mean())
        epoch_cluster = epoch_cluster[epoch_cluster["roi"].isin([seed])].reset_index(drop=True)
        plot_seed_eccentricity(epoch_cluster, seed, col)

        df_all = load_three_epochs(seed)
        df_all = df_all.merge(epoch_res[["roi_ix", "network"]], on="roi_ix", how="left")
        plot_polarplot_seed_diffs(df_all, seed=seed)
        
    for seed, col in zip(["7Networks_RH_Limbic_OFC_1", "7Networks_RH_SalVentAttn_FrOperIns_5" , "7Networks_LH_Default_Par_2", "7Networks_RH_DorsAttn_FEF_1"], ["#E1F8AD", "#B649F6", "#C1575D", "#337423"]):
        epoch_cluster = (epoch_res.groupby(['sub', 'roi', 'epoch', 'network'], as_index=False)['distance'].mean())
        epoch_cluster = epoch_cluster[epoch_cluster["roi"].isin([seed])].reset_index(drop=True)
        plot_seed_eccentricity(epoch_cluster, seed, col)

        df_all = load_three_epochs(seed)
        df_all = df_all.merge(epoch_res[["roi_ix", "network"]], on="roi_ix", how="left")
        plot_polarplot_seed_diffs(df_all, seed=seed)
        
main()